In [1]:
%load_ext autoreload
%autoreload 2

## 1. Imports

In [2]:
import os
import sys

sys.path.append("..")
sys.path.append("./ALAE")

import random

import numpy as np
import torch
from tqdm import tqdm

import wandb
from src.costs.lse import MLPLSECost
from src.models.gmm_based import GMMEOT
from src.models.light_sbm import LightSBM
from src.plotting.parameters import (
    plot_A_parameters,
    plot_B_parameters,
    plot_Z_parameters,
)
from src.samplers.from_dataset import DatasetSampler
from src.utils.train import compute_loss, update_average

In [3]:
device = torch.device(f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [4]:
torch.set_default_device(device)
# dtype = torch.float64
dtype = torch.float32
# torch.torch.set_default_dtype(dtype)

## 2. Config

In [5]:
from configs.gmm_based.cost import MLPLSECostConfig
from configs.gmm_based.optimizer import OptPairedConfig, OptUnpairedConfig
from configs.gmm_based.train import TrainConfig

In [6]:
# Data Type
X_DIM = 512
Y_DIM = 512
INPUT_DATA = "CHILDREN" # MAN, WOMAN, ADULT, CHILDREN
TARGET_DATA = "ADULT" # MAN, WOMAN, ADULT, CHILDREN

# Data
Q_X_UNPAIRED_SAMPLES = 48786 # 1024
R_Y_UNPAIRED_SAMPLES = 10762 # 1024
P_XY_PAIRED_SAMPLES = 5600 # 128

# Optimizer
LR_PAIRED = 1e-5
LR_UNPAIRED = 3e-4

# Sampler
PAIRED_BATCH_SIZE = 1024
UNPAIRED_BATCH_SIZE = 1024

# Train
MAX_STEPS = 10000
INIT_BY_SAMPLES = True

# Potential
N_POTENTIALS = 1

# Cost
M_POTENTIALS = 1
LOG_V_M_HIDDEN_CHANNELS = [M_POTENTIALS]
B_M_HIDDEN_CHANNELS = [M_POTENTIALS * Y_DIM]

In [7]:
cost_config = MLPLSECostConfig(
    x_dim=X_DIM,
    y_dim=Y_DIM,
    m_potentials=M_POTENTIALS,
    log_v_m_hidden_channels=LOG_V_M_HIDDEN_CHANNELS,
    b_m_hidden_channels=B_M_HIDDEN_CHANNELS,
)
EXP_META_INFO = (
    f"M_POTENTIALS_{M_POTENTIALS}_"
    + f"LOG_V_M_HIDDEN_CHANNELS_{LOG_V_M_HIDDEN_CHANNELS}_"
    + f"B_M_HIDDEN_CHANNELS_{B_M_HIDDEN_CHANNELS}_"
)

opt_unpaired_config = OptUnpairedConfig(lr=LR_UNPAIRED)
opt_paired_config = OptPairedConfig(lr=LR_PAIRED)

train_config = TrainConfig(
    steps_to=MAX_STEPS, paired_batch_size=PAIRED_BATCH_SIZE, unpaired_batch_size=UNPAIRED_BATCH_SIZE
)

In [8]:
torch.manual_seed(train_config.seed)
np.random.seed(train_config.seed)
random.seed(train_config.seed)

## 3. Create data and samplers

In [9]:
from src.utils.datasets import get_latents
from src.samplers.base import TensorSampler
from src.utils.paired import get_paired_sampler

In [10]:
X_train, X_test = get_latents(INPUT_DATA, dtype=dtype)
Y_train, Y_test = get_latents(TARGET_DATA, dtype=dtype)

In [11]:
X_sampler = TensorSampler(X_train.to(dtype), device=device)
Y_sampler = TensorSampler(Y_train.to(dtype), device=device)

In [12]:
from_dir = f"./datasets/FFHQ/pairs/{INPUT_DATA}->{TARGET_DATA}"
X_paired_train = torch.load(os.path.join(from_dir, f"X_train.pt"), map_location=device, weights_only=True).to(dtype)
Y_paired_train = torch.load(os.path.join(from_dir, f"Y_train.pt"), map_location=device, weights_only=True).to(dtype)

X_paired_test = torch.load(os.path.join(from_dir, f"X_test.pt"), map_location=device, weights_only=True).to(dtype)
Y_paired_test = torch.load(os.path.join(from_dir, f"Y_test.pt"), map_location=device, weights_only=True).to(dtype)

In [13]:
pd_train_sampler = get_paired_sampler(
    X_paired_train, Y_paired_train, train_config.paired_batch_size, P_XY_PAIRED_SAMPLES, device
)

In [14]:
P_XY_PAIRED_SAMPLES = min(P_XY_PAIRED_SAMPLES, X_paired_train.shape[0])
P_XY_PAIRED_SAMPLES

1273

In [15]:
Q_X_UNPAIRED_SAMPLES = min(Q_X_UNPAIRED_SAMPLES, X_train.shape[0])
Q_X_UNPAIRED_SAMPLES

10762

In [16]:
R_Y_UNPAIRED_SAMPLES = min(R_Y_UNPAIRED_SAMPLES, Y_train.shape[0])
R_Y_UNPAIRED_SAMPLES

10762

In [17]:
# if Q_X_UNPAIRED_SAMPLES > 0:
#     source_data = X_sampler.sample(Q_X_UNPAIRED_SAMPLES)
#     usd_sampler = DatasetSampler(source_data, device=device) # usd - unpaired source data
# else:
#     usd_sampler = DatasetSampler(X_paired_train, device=device)

# if R_Y_UNPAIRED_SAMPLES > 0:
#     target_data = Y_sampler.sample(R_Y_UNPAIRED_SAMPLES)
#     utd_sampler = DatasetSampler(target_data, device=device) # utd - unpaired target data
# else:
#     utd_sampler = DatasetSampler(Y_paired_train, device=device)

In [18]:
if Q_X_UNPAIRED_SAMPLES > 0:
    usd_sampler = DatasetSampler(X_train, device=device) # usd - unpaired source data
else:
    usd_sampler = DatasetSampler(X_paired_train, device=device)

if R_Y_UNPAIRED_SAMPLES > 0:
    utd_sampler = DatasetSampler(Y_train, device=device) # utd - unpaired target data
else:
    utd_sampler = DatasetSampler(Y_paired_train, device=device)

## 4. Model initialization

In [19]:
from src.costs.lse import BatchedLSECost

In [20]:
import torch.nn as nn
import torchvision

In [21]:
cost = MLPLSECost(**cost_config.model_dump())
# cost = BatchedLSECost(u, log_v_m_net, m_potentials=M_POTENTIALS)

In [22]:
model = GMMEOT(
    y_dim=Y_DIM,
    n_potentials=N_POTENTIALS,
    cost=cost,
).to(dtype)

if INIT_BY_SAMPLES:
    model.init_a_by_samples(Y_sampler.sample(N_POTENTIALS))

In [23]:
# For EMA update
if train_config.ema_update:
    model_copy = GMMEOT(
    y_dim=Y_DIM,
    n_potentials=N_POTENTIALS,
    cost=cost,
).to(dtype)

## 5. Optimizers initialization

In [24]:
unpaired_params_to_update = [model._log_w_n, model._a_n, model._log_A_n]

D_opt_unpaired = torch.optim.Adam(unpaired_params_to_update, **opt_unpaired_config.model_dump())

In [25]:
D_opt_paired = torch.optim.Adam(model.cost.parameters(), **opt_paired_config.model_dump())

In [26]:
# TODO: refactor this config
EXP_NAME = (
    "GMMEOT_ALAE_"
    + f"FROM_{INPUT_DATA}_"
    + f"TO_{TARGET_DATA}_"
    + f"P_XY_PAIRED_{P_XY_PAIRED_SAMPLES}_"
    + f"Q_X_UNPAIRED_{Q_X_UNPAIRED_SAMPLES}_"
    + f"R_Y_UNPAIRED_{R_Y_UNPAIRED_SAMPLES}_"
    + f"LR_PAIRED_{opt_paired_config.lr}_"
    + f"LR_UNPAIRED_{opt_unpaired_config.lr}_"
    + EXP_META_INFO
)
OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)

config = dict(
    D_LR_PAIRED=opt_paired_config.lr,
    D_LR_UNPAIRED=opt_unpaired_config.lr,
    BATCH_SIZE=train_config.unpaired_batch_size,
    P_XY_PAIRED_SAMPLES=P_XY_PAIRED_SAMPLES,
    Q_X_UNPAIRED_SAMPLES=Q_X_UNPAIRED_SAMPLES,
    R_Y_UNPAIRED_SAMPLES=R_Y_UNPAIRED_SAMPLES,
)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH, exist_ok=True)

In [27]:
if train_config.steps_from > 0:
    D_opt_unpaired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{train_config.steps_from}.pt")))
    D_opt_paired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_paired_{train_config.steps_from}.pt")))

## 6. Model training

In [28]:
starting_points = X_test[:3]
num_ending_points = 64

In [29]:
num_starting_points_paired = 5
indices = random.choices(range(P_XY_PAIRED_SAMPLES), k=num_starting_points_paired)
starting_points_paired = X_paired_train[indices]
ending_points_paired = Y_paired_train[indices]

In [30]:
wandb.init(name=EXP_NAME, config=config)

for step in tqdm(range(train_config.steps_from, train_config.steps_to)):
    # training loop
    D_opt_unpaired.zero_grad()

    X = usd_sampler.sample(train_config.unpaired_batch_size)
    Y = utd_sampler.sample(train_config.unpaired_batch_size)

    output_unpaired = model.compute_unpaired_loss(X, Y)
    D_loss_unpaired = output_unpaired["loss"]

    wandb.log({f"Unpaired loss": D_loss_unpaired.item()}, step=step)

    D_opt_paired.zero_grad()
    X_paired, Y_paired = pd_train_sampler.sample(train_config.paired_batch_size)
    
    output_paired = model.compute_paired_loss(X_paired, Y_paired)
    D_loss_paired = output_paired["loss"]

    wandb.log({f"Paired loss": D_loss_paired.item()}, step=step)

    D_loss = D_loss_unpaired + D_loss_paired
    D_loss.backward()
    D_opt_paired.step()
    D_opt_unpaired.step()

    if train_config.ema_update:
        update_average(model_copy, model, 0.99)
        model = model_copy
    else:
        model = model

    wandb.log({f"Loss": D_loss}, step=step)
    wandb.log(
        {f"Train paired loss": compute_loss(model, X_paired_train, Y_paired_train, X_paired_train, Y_paired_train)},
        step=step,
    )
    wandb.log(
        {f"Test paired loss": compute_loss(model, X_paired_test, Y_paired_test, X_paired_test, Y_paired_test)},
        step=step,
    )
    # wandb.log(
    #     {f"Test unpaired loss": compute_loss(model, X_unpaired_test, Y_unpaired_test, X_paired_test, Y_paired_test)},
    #     step=step,
    # )

    wandb.log({r"$-f^c(x)$": -output_unpaired["f_c"].mean().item()}, step=step)
    wandb.log({r"$-f(y)$": -output_unpaired["f"].mean().item()}, step=step)
    wandb.log({f"lam_min(A_n)": torch.min(output_unpaired["A_n"])}, step=step)
    wandb.log({f"lam_max(A_n)": torch.max(output_unpaired["A_n"])}, step=step)

    if step % train_config.plot_every == 0:
        # A_dict = plot_A_parameters(model, log=True)
        # B_dict = plot_B_parameters(model.cost, starting_points, log=True)
        # if num_starting_points_paired > 0:
        #     Z_dict = plot_Z_parameters(model, starting_points, starting_points_paired, ending_points_paired, log=True)
        # else:
        #     Z_dict = plot_Z_parameters(model, starting_points, log=True)
        # wandb.log(A_dict | B_dict | Z_dict, step=step)
        # wandb.log(A_dict | Z_dict, step=step)
        # wandb.log(Z_dict, step=step)
        # wandb.log(A_dict, step=step)

        torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"D_{step}.pt"))

torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"D_{MAX_STEPS}.pt"))
torch.save(D_opt_paired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_paired_{MAX_STEPS}.pt"))
torch.save(D_opt_unpaired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{MAX_STEPS}.pt"))

wandb.finish()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: muxaujl11110. Use `wandb login --relogin` to force relogin


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10000/10000 [04:47<00:00, 34.74it/s]


$-f(y)$,▂▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▇▇▇█
$-f^c(x)$,███████████████▇▇▇▇▇▇▇▇▆▆▆▅▅▅▅▅▅▅▄▅▄▄▃▂▁
Loss,███▇▇▆▆▆▅▅▄▄▄▄▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
Paired loss,█████▇▇▇▇▇▇▆▆▆▆▅▅▅▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▁▁
Test paired loss,█▇▆▄▃▃▃▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
Train paired loss,██▆▅▅▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▃▃▃
Unpaired loss,█▆▅▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▂▂▂▃▃▄▃▄▄▅▅▆▆▆▆▇▇███
lam_max(A_n),▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇█
lam_min(A_n),██▇▇▅▅▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
$-f(y)$,7579.13916
$-f^c(x)$,-5901.30322


# 7. Light-SBM training

In [31]:
import torch.nn.functional as F

In [32]:
eps = 0.1
lr = 1e-3

n_potentials = 10
is_diag = True
S_init = 0.1

max_iter = 20000

In [33]:
light_sbm = LightSBM(dim=X_DIM, n_potentials=n_potentials, epsilon=eps, S_diagonal_init=S_init, is_diagonal=is_diag)

light_sbm.to(device)
light_sbm_opt = torch.optim.Adam(light_sbm.parameters(), lr=lr)

In [79]:
def train(model, max_iter, eps, opt, val_freq=1000, batch_size=512, safe_t=1e-2, device=device):
    
    pbar = tqdm(range(1, max_iter + 1))
    
    for i in pbar:
        
        x_0_samples = X_sampler.sample(batch_size).to(device)      
        x_1_samples = Y_sampler.sample(batch_size).to(device)
        
        t = torch.rand([batch_size, 1], device=device) * (1 - safe_t)
        
        x_t = x_1_samples * t + x_0_samples * (1 - t) + torch.sqrt(eps * t * (1 - t)) * torch.randn_like(x_0_samples)
                
        predicted_drift = model.get_drift(x_t, t.squeeze())
        
        loss_plan = (model.get_log_C(x_0_samples) - model.get_log_potential(x_1_samples)).mean()
        
        target_drift = (x_1_samples - x_t) / (1 - t)
        
        loss = F.mse_loss(target_drift, predicted_drift)
        
        opt.zero_grad()
        
        loss.backward()
        
        opt.step()
        
        pbar.set_description(f'Loss : {loss.item()} Plan Loss: {loss_plan.item()}')
        
        if wandb.run:
            wandb.log({'loss_bm': loss, 'loss_plan': loss_plan})
        
        if i % val_freq == 0:
            pass

In [80]:
train(light_sbm, max_iter, eps, light_sbm_opt, val_freq=1000, batch_size=512, safe_t=1e-2, device=device)

  0%|                                                                                                                                                                                       | 0/20000 [00:00<?, ?it/s]/trinity/home/m.persiyanov/miniconda3/envs/text/lib/python3.11/site-packages/torch/utils/_device.py:106: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return func(*args, **kwargs)
Loss : 1.2854652404785156 Plan Loss: 4142.3828125: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20000/20000 [02:40<00:00, 124.59it/s]


# 8. Metrics

In [51]:
from alae_ffhq_inference import decode, load_model
from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

In [52]:
EVAL_MODEL_STEP = 10000

In [53]:
alae_model = load_model("./ALAE/configs/ffhq.yaml", training_artifacts_dir="./ALAE/training_artifacts/ffhq/").to(
    device
).to(dtype)

model.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_{EVAL_MODEL_STEP}.pt"), map_location=device, weights_only=True))

<All keys matched successfully>

In [34]:
OUTPUT_PATH

'../checkpoints/GMMEOT_ALAE_FROM_CHILDREN_TO_ADULT_P_XY_PAIRED_5600_Q_X_UNPAIRED_10762_R_Y_UNPAIRED_10762_LR_PAIRED_1e-05_LR_UNPAIRED_0.0003_M_POTENTIALS_1_LOG_V_M_HIDDEN_CHANNELS_[1]_B_M_HIDDEN_CHANNELS_[512]_'

In [35]:
def normalize_tensor(tensor: torch.Tensor) -> torch.Tensor:
    normalized = tensor / 2 + 0.5
    return normalized.clamp_(0, 1)

def to_uint8(normalized_tensor: torch.Tensor) -> torch.Tensor:
    return normalized_tensor.mul(255).add_(0.5).clamp_(0, 255).to(torch.uint8)

In [36]:
def eval_model(model: torch.nn.Module, model_name: str) -> tuple[float, float, float]:
    loss_fid = FrechetInceptionDistance().to(device)
    loss_ssim = StructuralSimilarityIndexMeasure(data_range=(-1.0, 1.0)).to(device)
    loss_lpip = LearnedPerceptualImagePatchSimilarity(net_type='alex').to(device)
    
    with torch.no_grad():
        sampling_batch_size = 64
        num_samples = len(X_paired_test)
        
        num_sampling_iterations = (
            num_samples // sampling_batch_size
            if num_samples % sampling_batch_size == 0
            else (num_samples // sampling_batch_size) + 1
        )
        for i in tqdm(range(num_sampling_iterations)):
            sub_batch_x = X_paired_test[sampling_batch_size * i : sampling_batch_size * (i + 1)]
            sub_batch_y = Y_paired_test[sampling_batch_size * i : sampling_batch_size * (i + 1)]

            y_pred = model(sub_batch_x)
            normalized_pred_images = normalize_tensor(decode(alae_model, y_pred))
            normalized_true_images = normalize_tensor(decode(alae_model, sub_batch_y))

            loss_fid.update(to_uint8(normalized_pred_images), real=True)
            loss_fid.update(to_uint8(normalized_true_images), real=False)

            loss_ssim.update(normalized_pred_images, normalized_true_images)
            loss_lpip.update(normalized_pred_images, normalized_true_images)

            # Explicitly free sub-batches to release GPU memory
            del sub_batch_x, sub_batch_y, y_pred, normalized_pred_images, normalized_true_images
            torch.cuda.empty_cache()
    
    loss_fid_out = loss_fid.compute()
    loss_ssim_out = loss_ssim.compute()
    loss_lpip_out = loss_lpip.compute()
    
    torch.save(loss_fid_out, os.path.join(OUTPUT_PATH, f"FID_{model_name}_{EVAL_MODEL_STEP}.pt"))
    torch.save(loss_ssim_out, os.path.join(OUTPUT_PATH, f"SSIM_{model_name}_{EVAL_MODEL_STEP}.pt"))
    torch.save(loss_lpip_out, os.path.join(OUTPUT_PATH, f"LPIP_{model_name}_{EVAL_MODEL_STEP}.pt"))
    
    return loss_fid_out, loss_ssim_out, loss_lpip_out

In [37]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [79]:
loss_fid, loss_ssim, loss_lpip = eval_model(model,  "our")
print(f"FID: {loss_fid}")
print(f"SSIM: {loss_ssim}")
print(f"LPIPS: {loss_lpip}")

/trinity/home/m.persiyanov/miniconda3/envs/text/lib/python3.11/site-packages/torchmetrics/functional/image/lpips.py:323: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.l

FID: 73.62894439697266
SSIM: 0.5127382874488831
LPIPS: 0.4716850519180298


In [94]:
loss_fid_light_sbm, loss_ssim_light_sbm, loss_lpip_light_sbm = eval_model(light_sbm, "light-sbm")
print(f"FID: {loss_fid_light_sbm}")
print(f"SSIM: {loss_ssim_light_sbm}")
print(f"LPIPS: {loss_lpip_light_sbm}")

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:15<00:00,  1.72s/it]


FID: 23.10447120666504
SSIM: 0.7093340158462524
LPIPS: 0.22459906339645386


# 8. Plotting

In [54]:
import numpy as np
import matplotlib.pyplot as plt
import torch

In [55]:
# Parameters
num_images = 20       # Number of test images to show
num_gen = 1           # Number of generated versions per image
models = [model] # [light_sbm, model]  # List of models to evaluate
model_names = ['Ours'] # ['Light-SBM', 'Ours']  # For titles

# Select random test samples
indices = np.random.choice(len(X_paired_test), num_images, replace=False)
x = X_paired_test[indices]
y = Y_paired_test[indices]

# Decode input and target images
init_img = normalize_tensor(decode(alae_model, x))
true_img = normalize_tensor(decode(alae_model, y))

# Generate predictions for each model
all_model_preds = []  # List to hold predictions from each model

for model in models:
    model_preds = []
    for _ in range(num_gen):
        with torch.no_grad():
            y_pred = model(x)
            decoded = normalize_tensor(decode(alae_model, y_pred))
            model_preds.append(decoded)
    model_preds = torch.stack(model_preds, dim=1)  # [num_images, num_gen, C, H, W]
    all_model_preds.append(model_preds)

# Convert to numpy arrays for plotting
init_img_np = init_img.cpu().permute(0, 2, 3, 1).numpy()
true_img_np = true_img.cpu().permute(0, 2, 3, 1).numpy()
all_model_preds_np = [
    preds.cpu().permute(0, 1, 3, 4, 2).numpy()
    for preds in all_model_preds
]

In [56]:
# Plotting
cols = 2 + num_gen * len(models)
fig, axes = plt.subplots(
    num_images,
    cols,
    figsize=(cols, num_images * 1.2),  # Reduced row height
    dpi=200
)

for i in range(num_images):
    # Input image
    axes[i, 0].imshow(init_img_np[i])
    axes[i, 0].set_title('Input' if i == 0 else '')
    axes[i, 0].axis('off')

    # Target image
    axes[i, 1].imshow(true_img_np[i])
    axes[i, 1].set_title('Target' if i == 0 else '')
    axes[i, 1].axis('off')

    # Generated images
    col_idx = 2
    for m_idx, model_preds in enumerate(all_model_preds_np):
        for g_idx in range(num_gen):
            axes[i, col_idx].imshow(model_preds[i, g_idx])
            if i == 0:
                axes[i, col_idx].set_title(f'{model_names[m_idx]}')
            axes[i, col_idx].axis('off')
            col_idx += 1

# Tighter layout
plt.tight_layout(pad=0.1)  # Smaller padding between rows/columns
plt.subplots_adjust(hspace=0.02)  # Even tighter vertical spacing

plt.savefig(f'{INPUT_DATA}->{TARGET_DATA}_full.png', bbox_inches='tight')
plt.close()

In [84]:
selected_indices = {0:1, 1:4, 2:5, 3:6, 4:7, 5:8}

In [64]:
# Plotting
cols = 2 + num_gen * len(models)
fig, axes = plt.subplots(
    len(selected_indices),
    cols,
    figsize=(cols, len(selected_indices) * 1.5),
    dpi=200
)

for i in range(len(selected_indices)):
    # Input image
    j = selected_indices[i]
    axes[i, 0].imshow(init_img_np[j])
    axes[i, 0].set_title('Input' if i == 0 else '')
    axes[i, 0].axis('off')

    # Target image
    axes[i, 1].imshow(true_img_np[j])
    axes[i, 1].set_title('Target' if i == 0 else '')
    axes[i, 1].axis('off')

    # Generated images from each model
    col_idx = 2
    for m_idx, model_preds in enumerate(all_model_preds_np):
        for g_idx in range(num_gen):
            axes[i, col_idx].imshow(model_preds[j, g_idx])
            if i == 0:
                axes[i, col_idx].set_title(f'{model_names[m_idx]}')
            axes[i, col_idx].axis('off')
            col_idx += 1

plt.tight_layout(pad=0.5)
plt.savefig(f'{INPUT_DATA}->{TARGET_DATA}_selected.png', bbox_inches='tight')
plt.close()